只保留替换突变，8060个数据

In [3]:
import pandas as pd
import re

def process_mutational_data(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    
    if 'Variant (one letter)' not in df.columns or 'Cum_score' not in df.columns:
        raise ValueError("CSV 文件中缺少 'Variant (one letter)' 或 'Cum_score' 列。")

    wt_pattern = re.compile(r'^([A-Za-z])(\d+)')
    
    wt_dict = {}
    for variant in df['Variant (one letter)'].dropna():
        match = wt_pattern.match(variant)
        if match:
            wt_aa = match.group(1)
            pos = int(match.group(2))
            wt_dict[pos] = wt_aa

    wt_seq_list = [wt_dict.get(i, 'X') for i in range(1, 404)]

    substitution_pattern = r'^[A-Za-z]\d+[A-Za-z]$'
    df_filtered = df[df['Variant (one letter)'].str.match(substitution_pattern, na=False)].copy()

    parse_pattern = re.compile(r'^([A-Za-z])(\d+)([A-Za-z])$')

    def generate_mutated_seq(variant):
        match = parse_pattern.match(variant)
        if not match:
            return None 
            
        pos = int(match.group(2))
        mut_aa = match.group(3)
        
        current_seq = wt_seq_list.copy()
        
        if 1 <= pos <= 403:
            current_seq[pos - 1] = mut_aa
                
        return "".join(current_seq)

    df_filtered['mutant'] = df_filtered['Variant (one letter)']
    df_filtered['mutated_sequence'] = df_filtered['mutant'].apply(generate_mutated_seq)
    
    final_df = df_filtered[['mutant', 'mutated_sequence', 'Cum_score']]
    
    final_df.to_csv(output_csv, index=False)
    
    print(f"野生型序列共收集到 {len(wt_dict)} 个位置的信息。")
    print(f"原始数据总行数: {len(df)}")
    print(f"过滤后（仅保留替换突变）行数: {len(final_df)}")
    print(f"删除了 {len(df) - len(final_df)} 行非替换突变数据。")
    print(f"结果已保存至: {output_csv}")
    
    return final_df

process_mutational_data('/share/home/wangtb/PTEN_data/data.csv', '/share/home/wangtb/enzyme_shells/activity_data/PTEN.csv')

野生型序列共收集到 403 个位置的信息。
原始数据总行数: 8866
过滤后（仅保留替换突变）行数: 8060
删除了 806 行非替换突变数据。
结果已保存至: /share/home/wangtb/enzyme_shells/activity_data/PTEN.csv


,mutant,mutated_sequence,Cum_score
0,M1A,ATAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
1,M1V,VTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
2,M1I,ITAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,-0.679813
3,M1L,LTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,-1.545614
4,M1M,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
...,...,...,...
8859,V403R,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
8860,V403H,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
8861,V403K,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN
8862,V403D,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...,NaN


看看ProteinGym的PTEN数据序列跟我们的一样不一样，发现完全一样，为了对比，我们把这个也跑一下

In [4]:
import pandas as pd
import re

def extract_wt_sequence(csv_path, variant_col_candidates=None):

    if variant_col_candidates is None:
        variant_col_candidates = ['Variant (one letter)', 'mutant', 'variant', 'Mutation']
        
    df = pd.read_csv(csv_path)
    
    target_col = None
    for col in variant_col_candidates:
        if col in df.columns:
            target_col = col
            break
            
    if not target_col:
        raise ValueError(f"在 {csv_path} 中找不到指定的突变列，请检查列名。")

    wt_pattern = re.compile(r'^([A-Za-z])(\d+)')
    wt_dict = {}
    
    for variant in df[target_col].dropna():
        match = wt_pattern.match(str(variant).strip())
        if match:
            wt_aa = match.group(1).upper()
            pos = int(match.group(2))
            
            if pos in wt_dict and wt_dict[pos] != wt_aa:
                print(f"警告：文件 {csv_path} 中位置 {pos} 存在冲突的野生型氨基酸 ({wt_dict[pos]} vs {wt_aa})")
                
            wt_dict[pos] = wt_aa

    if not wt_dict:
        return ""

    max_pos = max(wt_dict.keys())
    wt_seq_list = [wt_dict.get(i, 'X') for i in range(1, max_pos + 1)]
    
    return "".join(wt_seq_list)


def compare_sequences(file1, file2):
    print(f"正在解析文件 1: {file1}")
    seq1 = extract_wt_sequence(file1)
    
    print(f"正在解析文件 2: {file2}")
    seq2 = extract_wt_sequence(file2)
    
    print("-" * 50)
    print(f"文件 1 序列长度: {len(seq1)}")
    print(f"文件 2 序列长度: {len(seq2)}")
    
    if seq1 == seq2:
        print("\n✅ 结论：两个文件的原始序列完全一致！")
    else:
        print("\n❌ 结论：两个文件的原始序列不一致！")
        
        if len(seq1) != len(seq2):
            print(f"-> 序列长度不同！")
            
        min_len = min(len(seq1), len(seq2))
        mismatches = []
        for i in range(min_len):
            if seq1[i] != seq2[i]:
                mismatches.append(f"位置 {i+1}: 文件1为 '{seq1[i]}' , 文件2为 '{seq2[i]}'")
                
        if mismatches:
            print(f"-> 在前 {min_len} 个残基中，发现 {len(mismatches)} 处不同。以下是前 10 处差异：")
            for mismatch in mismatches[:10]:
                print(f"   {mismatch}")
            if len(mismatches) > 10:
                print("   ...")

if __name__ == "__main__":
    file_path_1 = "/share/home/wangtb/enzyme_shells/activity_data/PTEN_HUMAN_Mighell_2018.csv"
    file_path_2 = "/share/home/wangtb/enzyme_shells/activity_data/PTEN.csv"
    
    compare_sequences(file_path_1, file_path_2)

正在解析文件 1: /share/home/wangtb/enzyme_shells/activity_data/PTEN_HUMAN_Mighell_2018.csv
正在解析文件 2: /share/home/wangtb/enzyme_shells/activity_data/PTEN.csv
--------------------------------------------------
文件 1 序列长度: 403
文件 2 序列长度: 403

✅ 结论：两个文件的原始序列完全一致！


看看提供的预测结构序列是否一致，发现完全一致，为了稳妥，我们采用同一个预测结构序列进行对比分析。（PTEN）

In [9]:
from Bio import PDB
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

def get_sequence_from_structure(file_path):
    file_ext = file_path.split('.')[-1].lower()
    
    if file_ext == 'pdb':
        parser = PDB.PDBParser(QUIET=True)
    elif file_ext in ['cif', 'mmcif']:
        parser = PDB.MMCIFParser(QUIET=True)
    else:
        raise ValueError(f"不支持的文件格式: {file_ext}")

    structure = parser.get_structure("protein", file_path)
    sequences = {}

    for model in structure:
        for chain in model:
            chain_id = chain.get_id()
            seq_list = []

            for residue in chain:
                # 过滤非标准氨基酸 + HETATM
                if is_aa(residue, standard=True):
                    resname = residue.get_resname().strip()

                    try:
                        # seq1 比 three_to_one 更稳
                        aa = seq1(resname)
                    except Exception:
                        aa = 'X'

                    seq_list.append(aa)

            if seq_list:
                sequences[chain_id] = "".join(seq_list)

        break  # 只取第一个 model
        
    return sequences


def compare_structural_sequences(file_pdb, file_cif):
    print(f"正在读取 PDB: {file_pdb}")
    seqs_pdb = get_sequence_from_structure(file_pdb)
    
    print(f"正在读取 CIF: {file_cif}")
    seqs_cif = get_sequence_from_structure(file_cif)
    
    print("\n" + "="*40)
    print("序列对比结果")
    print("="*40)

    all_chains = set(seqs_pdb.keys()).union(set(seqs_cif.keys()))
    
    for cid in sorted(all_chains):
        print(f"\n--- Chain {cid} ---")
        s1 = seqs_pdb.get(cid, None)
        s2 = seqs_cif.get(cid, None)
        
        if s1 is None:
            print(f"结果: PDB 文件中不存在 Chain {cid}")
            continue
        if s2 is None:
            print(f"结果: CIF 文件中不存在 Chain {cid}")
            continue
            
        print(f"PDB 序列长度: {len(s1)}")
        print(f"CIF 序列长度: {len(s2)}")
        
        if s1 == s2:
            print("状态: ✅ 完全一致")
        else:
            print("状态: ❌ 存在差异")
            min_len = min(len(s1), len(s2))
            diffs = [i+1 for i in range(min_len) if s1[i] != s2[i]]

            if diffs:
                print(f"不匹配的位置 (前10个): {diffs[:10]}{'...' if len(diffs)>10 else ''}")

            if len(s1) != len(s2):
                print("提示: 序列长度不等，可能是由于结构中缺失 loop 或 unresolved 区域。")


# ==== 测试 ====
pdb_file = "/share/home/wangtb/enzyme_shells/structure/PTEN_2018.pdb"
cif_file = "/share/home/wangtb/enzyme_shells/structure/PTEN.cif"
    
try:
    compare_structural_sequences(pdb_file, cif_file)
except Exception as e:
    print(f"运行出错: {e}")

正在读取 PDB: /share/home/wangtb/enzyme_shells/structure/PTEN_2018.pdb
正在读取 CIF: /share/home/wangtb/enzyme_shells/structure/PTEN.cif

序列对比结果

--- Chain A ---
PDB 序列长度: 403
CIF 序列长度: 403
状态: ✅ 完全一致


看看fasta和结构是否一一对应

In [10]:
import os
from Bio import SeqIO
from Bio import PDB
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1


def get_sequence_from_structure(file_path):
    file_ext = file_path.split('.')[-1].lower()

    if file_ext == 'pdb':
        parser = PDB.PDBParser(QUIET=True)
    elif file_ext in ['cif', 'mmcif']:
        parser = PDB.MMCIFParser(QUIET=True)
    else:
        return None

    try:
        structure = parser.get_structure("protein", file_path)
        sequences = {}

        for model in structure:
            for chain in model:
                chain_id = chain.get_id()
                seq_list = []

                for residue in chain:
                    # 过滤非标准氨基酸 & HETATM
                    if is_aa(residue, standard=True):
                        resname = residue.get_resname().strip()

                        try:
                            aa = seq1(resname)
                        except Exception:
                            aa = 'X'

                        seq_list.append(aa)

                if seq_list:
                    sequences[chain_id] = "".join(seq_list)

            break  # 只取第一个 model

        return sequences

    except Exception as e:
        print(f"解析结构文件 {file_path} 出错: {e}")
        return None


def get_sequence_from_fasta(file_path):
    try:
        record = list(SeqIO.parse(file_path, "fasta"))
        if record:
            return str(record[0].seq)
        return None
    except Exception as e:
        print(f"解析 FASTA 文件 {file_path} 出错: {e}")
        return None


def batch_compare(fasta_dir, struct_dir):
    fasta_files = {
        os.path.splitext(f)[0]: f
        for f in os.listdir(fasta_dir)
        if f.endswith('.fasta')
    }

    struct_files = {}
    for f in os.listdir(struct_dir):
        if f.endswith(('.pdb', '.cif')):
            name = os.path.splitext(f)[0]
            struct_files[name] = f

    print(f"{'Index':<20} | {'Status':<15} | {'Details'}")
    print("-" * 70)

    for name, f_file in fasta_files.items():
        if name in struct_files:
            s_file = struct_files[name]

            f_seq = get_sequence_from_fasta(os.path.join(fasta_dir, f_file))
            s_seqs = get_sequence_from_structure(os.path.join(struct_dir, s_file))

            if f_seq and s_seqs:
                match_found = False
                matched_chain = None

                for cid, s_seq in s_seqs.items():
                    if f_seq == s_seq:
                        match_found = True
                        matched_chain = cid
                        break

                if match_found:
                    print(f"{name:<20} | ✅ Match       | FASTA matches Chain {matched_chain}")
                else:
                    s_len_info = "/".join([str(len(v)) for v in s_seqs.values()])
                    print(f"{name:<20} | ❌ Mismatch    | FASTA len: {len(f_seq)}, Struct chains len: {s_len_info}")
            else:
                print(f"{name:<20} | ⚠️ Error       | Could not parse sequences")
        else:
            print(f"{name:<20} | 🔍 Missing     | No matching structure file found")


# ==== 路径 ====
fasta_folder = "/share/home/wangtb/enzyme_shells/fasta"
structure_folder = "/share/home/wangtb/enzyme_shells/structure"

if __name__ == "__main__":
    batch_compare(fasta_folder, structure_folder)

Index                | Status          | Details
----------------------------------------------------------------------
AICDA                | ✅ Match       | FASTA matches Chain A
AMIE                 | ✅ Match       | FASTA matches Chain A
CAS9                 | ✅ Match       | FASTA matches Chain A
LGK                  | ✅ Match       | FASTA matches Chain A
OTC                  | ✅ Match       | FASTA matches Chain A
PTEN_2018            | ✅ Match       | FASTA matches Chain A
RASH                 | ✅ Match       | FASTA matches Chain A
RNC                  | ✅ Match       | FASTA matches Chain A
RUBISCO              | ✅ Match       | FASTA matches Chain A
VKOR1                | ✅ Match       | FASTA matches Chain A
PAFA                 | ✅ Match       | FASTA matches Chain A


看看csv的突变是否和fasta的突变对应位置是一致的

In [12]:
import os
import re
from Bio import SeqIO


def load_fasta_sequence(fasta_path):
    try:
        record = next(SeqIO.parse(fasta_path, "fasta"))
        return str(record.seq)
    except Exception as e:
        print(f"[FASTA ERROR] {fasta_path}: {e}")
        return None


def parse_mutation(mut_str):
    """
    解析突变字符串，例如:
    A23B → ('A', 23, 'B')
    支持多突变：A23B;C45D 或 A23B/C45D
    """
    if not isinstance(mut_str, str):
        return []

    mut_str = mut_str.strip()

    # 分割多突变
    parts = re.split(r"[;/]", mut_str)

    results = []
    for p in parts:
        p = p.strip()
        match = re.match(r"([A-Z])(\d+)([A-Z])", p)
        if match:
            wt, pos, mut = match.groups()
            results.append((wt, int(pos), mut))
    return results


def find_best_offset(fasta_seq, mutations, offset_range=50):
    best_offset = 0
    best_score = -1

    for offset in range(-offset_range, offset_range + 1):
        score = 0

        for wt, pos, mut in mutations:
            new_pos = pos + offset

            if 1 <= new_pos <= len(fasta_seq):
                if fasta_seq[new_pos - 1] == wt:
                    score += 1

        if score > best_score:
            best_score = score
            best_offset = offset

    return best_offset, best_score


def process_csv_file(csv_path, fasta_seq):
    import pandas as pd

    df = pd.read_csv(csv_path)

    if "mutant" not in df.columns:
        print(f"[SKIP] {csv_path} 没有 mutant 列")
        return

    all_mutations = []

    for _, row in df.iterrows():
        muts = parse_mutation(row["mutant"])
        all_mutations.extend(muts)

    if not all_mutations:
        print(f"{os.path.basename(csv_path):<15} | ❌ 无突变数据")
        return

    # 🔥 自动找 offset
    best_offset, best_score = find_best_offset(fasta_seq, all_mutations)

    total = len(all_mutations)
    ok = best_score
    mismatch = total - ok

    print(f"{os.path.basename(csv_path):<15} | Offset: {best_offset:+4d} | OK: {ok:<5} | Mismatch: {mismatch:<5}")


def batch_check(fasta_dir, csv_dir):
    fasta_files = {
        os.path.splitext(f)[0]: f
        for f in os.listdir(fasta_dir)
        if f.endswith(".fasta")
    }

    csv_files = {
        os.path.splitext(f)[0]: f
        for f in os.listdir(csv_dir)
        if f.endswith(".csv")
    }

    print(f"{'Name':<15} | Summary")
    print("-" * 70)

    for name, fasta_file in fasta_files.items():
        if name in csv_files:
            fasta_path = os.path.join(fasta_dir, fasta_file)
            csv_path = os.path.join(csv_dir, csv_files[name])

            fasta_seq = load_fasta_sequence(fasta_path)

            if fasta_seq:
                process_csv_file(csv_path, fasta_seq)
            else:
                print(f"{name:<15} | FASTA 读取失败")
        else:
            print(f"{name:<15} | ❌ 没有对应 CSV")


fasta_dir = "/share/home/wangtb/enzyme_shells/fasta"
csv_dir = "/share/home/wangtb/enzyme_shells/activity_data"

batch_check(fasta_dir, csv_dir)

Name            | Summary
----------------------------------------------------------------------
AICDA.csv       | Offset:   +0 | OK: 209   | Mismatch: 0    
AMIE.csv        | Offset:   +0 | OK: 6227  | Mismatch: 0    
CAS9.csv        | Offset:   +0 | OK: 8117  | Mismatch: 0    
LGK.csv         | Offset:   +0 | OK: 7890  | Mismatch: 0    
OTC.csv         | Offset:   +0 | OK: 1570  | Mismatch: 0    
PTEN_2018.csv   | Offset:   +0 | OK: 7260  | Mismatch: 0    
RASH.csv        | Offset:   +0 | OK: 3134  | Mismatch: 0    
RNC.csv         | Offset:   +0 | OK: 4277  | Mismatch: 0    
RUBISCO.csv     | Offset:   -1 | OK: 8694  | Mismatch: 75   
VKOR1.csv       | Offset:   +0 | OK: 697   | Mismatch: 0    
PAFA.csv        | Offset:   +0 | OK: 1018  | Mismatch: 0    


In [13]:
import re
import pandas as pd
from Bio import SeqIO


# ==== 1. 读取 FASTA ====
def load_fasta_sequence(fasta_path):
    record = next(SeqIO.parse(fasta_path, "fasta"))
    return str(record.seq)


# ==== 2. 解析 mutation ====
def parse_mutation(mut_str):
    if not isinstance(mut_str, str):
        return []

    parts = re.split(r"[;/]", mut_str.strip())
    results = []

    for p in parts:
        m = re.match(r"([A-Z])(\d+)([A-Z])", p.strip())
        if m:
            wt, pos, mut = m.groups()
            results.append((wt, int(pos), mut))

    return results


# ==== 3. 局部窗口检查 ====
def local_check(fasta_seq, wt, pos, offset, window=2):
    new_pos = pos + offset

    for shift in range(-window, window + 1):
        p = new_pos + shift
        if 1 <= p <= len(fasta_seq):
            if fasta_seq[p - 1] == wt:
                return True, p
    return False, None


# ==== 4. 主分析函数 ====
def analyze_rubisco(csv_path, fasta_path, offset=-1, out_file="rubisco_mismatch_analysis.csv"):
    fasta_seq = load_fasta_sequence(fasta_path)
    df = pd.read_csv(csv_path)

    results = []

    for idx, row in df.iterrows():
        muts = parse_mutation(row["mutant"])

        for wt, pos, mut in muts:
            new_pos = pos + offset

            # 越界
            if not (1 <= new_pos <= len(fasta_seq)):
                results.append({
                    "mutation": f"{wt}{pos}{mut}",
                    "corrected_pos": new_pos,
                    "fasta_aa": None,
                    "status": "OUT_OF_RANGE"
                })
                continue

            fasta_aa = fasta_seq[new_pos - 1]

            if fasta_aa == wt:
                continue  # 正确的我们不关心

            # 局部检查
            is_local, local_pos = local_check(fasta_seq, wt, pos, offset)

            if is_local:
                status = "LOCAL_MATCH"
            else:
                status = "TRUE_MISMATCH"

            results.append({
                "mutation": f"{wt}{pos}{mut}",
                "corrected_pos": new_pos,
                "fasta_aa": fasta_aa,
                "status": status,
                "local_pos": local_pos
            })

    result_df = pd.DataFrame(results)

    # 保存
    result_df.to_csv(out_file, index=False)

    # 打印 summary
    print("\n===== RUBISCO mismatch breakdown =====")
    print(result_df["status"].value_counts())

    print(f"\n详细结果已保存到: {out_file}")


# ==== 路径 ====
csv_path = "/share/home/wangtb/enzyme_shells/activity_data/RUBISCO.csv"
fasta_path = "/share/home/wangtb/enzyme_shells/fasta/RUBISCO.fasta"

if __name__ == "__main__":
    analyze_rubisco(csv_path, fasta_path)


===== RUBISCO mismatch breakdown =====
status
OUT_OF_RANGE    75
Name: count, dtype: int64

详细结果已保存到: rubisco_mismatch_analysis.csv


可以看到，是rubisco的一端被截断了，导致csv范围大于fasta，但是整体上突变位置是一致的，所以我们可以继续进行对比分析。